In [1]:
import pandas as pd
import gzip
import json
from pm4py.utils import format_dataframe
from pm4py import write_xes
from tqdm.notebook import tqdm
from copy import deepcopy

In [23]:
dataset_json_path = r".out/eventlogs/xbpic17-0.3-1.json.gz"
new_dataset_json_path = r".out/eventlogs/bpic17-0.3-1.json.gz"
with gzip.open(dataset_json_path, "r") as f:
        data = f.read()
        j = json.loads(data.decode('utf-8'))
# j_sampled = deepcopy(j)


In [24]:
print(j["cases"][3]["attributes"]) 
print(j["cases"][3]["events"][0]) 

"""
case

{'AMOUNT_REQ': '20000',
'REG_DATE': '2011-10-01T00:38:44.546+02:00',
'concept:name': '173688',
'label': 'normal'}

{'concept:name': 'Accepted', 
'impact': 'Low', 
'lifecycle:transition': 'In Progress', 
'org:group': 'Org line A2',
'org:resource': 'Tomas', 
'org:role': 'A2_2', 
'organization country': 'cn', 
'organization involved': 'M1 2nd', 
'product': 'PROD753', 
'resource country': 'Sweden',
'time:timestamp': '2007-05-10T16:21:54+02:00'}, 
'name': 'Accepted+In Progress',
'timestamp': '2007-05-10T16:21:54+02:00', 
'timestamp_end': None}

{'action_code': '01_HOOFD_010',
'activityNameEN': 'register submission date request',
'activityNameNL': 'registratie datum binnenkomst aanvraag',
'concept:name': '01_HOOFD_010', 
'dateFinished': '2014-04-25 00:00:00',
'dueDate': '2014-04-28T10:46:46+02:00',
'lifecycle:transition': 'complete',
'monitoringResource': '560912',
'org:resource': '560872',
'planned': '2014-04-24T10:46:46+02:00',
'question': 'EMPTY',
'time:timestamp': '2014-04-17T00:00:00+02:00'},
'name': '01_HOOFD_010+complete', 
'timestamp': '2014-04-17T00:00:00+02:00',
'timestamp_end': None}

{'attributes': {'Action': 'Created', 'EventID': 'Application_1746793196', 'EventOrigin': 'Application', 'concept:name': 'A_Create Application', 'lifecycle:transition': 'complete', 'org:resource': 'User_1', 'time:timestamp': '2016-01-01T12:34:53.911Z'}, 'name': 'A_Create Application', 'timestamp': '2016-01-01T12:34:53.911Z', 'timestamp_end': None}


"""
# check how many of the cases are normal and how many are anomlay
normal = 0
anomaly = 0
for case in j["cases"]:
    if case["attributes"]["label"] == "normal":
        normal += 1
    else:
        anomaly += 1
print(f"normal: {normal}, anomaly: {anomaly}")

{'ApplicationType': 'New credit', 'LoanGoal': 'Car', 'RequestedAmount': '5000.0', 'concept:name': 'Application_1746793196', 'label': {'anomaly': 'Attribute', 'attr': {'attribute': ['org:resource', 'org:resource', 'org:resource'], 'attribute_index': [0, 0, 0], 'index': [21, 46, 49], 'original': ['User_19', 'User_114', 'User_114']}}}
{'attributes': {'Action': 'Created', 'EventID': 'Application_1746793196', 'EventOrigin': 'Application', 'concept:name': 'A_Create Application', 'lifecycle:transition': 'complete', 'org:resource': 'User_1', 'time:timestamp': '2016-01-01T12:34:53.911Z'}, 'name': 'A_Create Application', 'timestamp': '2016-01-01T12:34:53.911Z', 'timestamp_end': None}
normal: 22044, anomaly: 9465


In [25]:
print(j["cases"][3]["attributes"]) #{'ApplicationType': 'New credit', 'LoanGoal': 'Car', 'RequestedAmount': '5000.0', 'concept:name': 'Application_1746793196', 'label': {'anomaly': 'Attribute', 'attr': {'attribute': ['org:resource', 'org:resource', 'org:resource'], 'attribute_index': [0, 0, 0], 'index': [21, 46, 49], 'original': ['User_19', 'User_114', 'User_114']}}}
print(j["cases"][3]["events"][0]) #{'attributes': {'Action': 'Created', 'EventID': 'Application_1746793196', 'EventOrigin': 'Application', 'concept:name': 'A_Create Application', 'lifecycle:transition': 'complete', 'org:resource': 'User_1', 'time:timestamp': '2016-01-01T12:34:53.911Z'}, 'name': 'A_Create Application', 'timestamp': '2016-01-01T12:34:53.911Z', 'timestamp_end': None}

def remove_specific_keys_in_event_attributes(event):
    keys_to_remove = ["Action", "EventID", "EventOrigin", "lifecycle:transition"]
    for key in keys_to_remove:
        event["attributes"].pop(key, None)
    # event.pop("timestamp_end", None)
    return event

def remove_specific_keys_in_case_attributes(case):
    keys_to_remove = ["ApplicationType", "LoanGoal", "RequestedAmount"]
    for key in keys_to_remove:
        case["attributes"].pop(key, None)
    return case

def remove_keys_in_case_events(case):
    case["events"] = [remove_specific_keys_in_event_attributes(event) for event in case["events"]]
    return case

def modify_case(case):
    case = remove_specific_keys_in_case_attributes(case)
    case = remove_keys_in_case_events(case)
    return case

def modify_event_log_j(j_orig):
    j_modified = deepcopy(j_orig)
    j_modified["cases"] = [modify_case(case) for case in j_modified["cases"]]
    return j_modified

j_new = modify_event_log_j(j)

print(j_new["cases"][3]["attributes"])
print(j_new["cases"][3]["events"][0])


{'ApplicationType': 'New credit', 'LoanGoal': 'Car', 'RequestedAmount': '5000.0', 'concept:name': 'Application_1746793196', 'label': {'anomaly': 'Attribute', 'attr': {'attribute': ['org:resource', 'org:resource', 'org:resource'], 'attribute_index': [0, 0, 0], 'index': [21, 46, 49], 'original': ['User_19', 'User_114', 'User_114']}}}
{'attributes': {'Action': 'Created', 'EventID': 'Application_1746793196', 'EventOrigin': 'Application', 'concept:name': 'A_Create Application', 'lifecycle:transition': 'complete', 'org:resource': 'User_1', 'time:timestamp': '2016-01-01T12:34:53.911Z'}, 'name': 'A_Create Application', 'timestamp': '2016-01-01T12:34:53.911Z', 'timestamp_end': None}
{'concept:name': 'Application_1746793196', 'label': {'anomaly': 'Attribute', 'attr': {'attribute': ['org:resource', 'org:resource', 'org:resource'], 'attribute_index': [0, 0, 0], 'index': [21, 46, 49], 'original': ['User_19', 'User_114', 'User_114']}}}
{'attributes': {'concept:name': 'A_Create Application', 'org:res

In [26]:
import numpy as np

# Get the total number of cases
orig_num_cases = len(j_new["cases"])
print(f"Total number of orig cases: {orig_num_cases}")

# Sample 6000 cases or all cases if less than 6000
# sample_size = min(6000, num_cases)
# sampled_indices = np.random.choice(num_cases, sample_size, replace=False)

# Create j_sampled with the sampled cases
j_sampled = deepcopy(j_new)

# select only cases whole length is less than 15
j_sampled["cases"] = [j_new["cases"][i] for i in range(orig_num_cases) if len(j_new["cases"][i]["events"]) < 25]

new_num_cases = len(j_sampled["cases"])
print(f"Number of cases after filtering: {new_num_cases}")
sample_size = new_num_cases
sampled_indices = np.random.choice(new_num_cases, sample_size, replace=False)

j_sampled["cases"] = [j_sampled["cases"][i] for i in sampled_indices]

print(f"Number of final sampled cases: {len(j_sampled['cases'])}")
normal = 0
anomaly = 0
for case in j_sampled["cases"]:
    if case["attributes"]["label"] == "normal":
        normal += 1
    else:
        anomaly += 1
print(f"normal: {normal}, anomaly: {anomaly}")


Total number of orig cases: 31509
Number of cases after filtering: 6801
Number of final sampled cases: 6801
normal: 4854, anomaly: 1947


In [27]:

# Save the sampled data to a new file
with gzip.open(new_dataset_json_path, "w") as f:
    f.write(json.dumps(j_sampled).encode('utf-8'))


In [28]:
def binet_to_df(path):
    with gzip.open(path, "r") as f:
        data = f.read()
        j = json.loads(data.decode('utf-8'))
    
    res_list = []
    
    for case in j['cases']:
        trace = pd.DataFrame.from_dict(case['events'])
        trace['anomaly'] = case['attributes']['label'] if isinstance(case['attributes']['label'], str) else case['attributes']['label']['anomaly']
        trace['trace_id'] = case['id']
        res_list.append(trace)
    
    if res_list:
        res = pd.concat(res_list, ignore_index=True)
        res = pd.concat([res.drop(['attributes'], axis=1), res['attributes'].apply(pd.Series)], axis=1)
    else:
        res = pd.DataFrame()
    
    return res

from datetime import datetime, timedelta

def assign_sequential_timestamps(df, start_time=None, step_minutes=10, duration_minutes=5):
    if start_time is None:
        start_time = datetime.now()

    df = df.copy()
    timestamps = []
    timestamps_end = []

    for trace_id, group in df.groupby('trace_id'):
        base_time = start_time
        for _ in range(len(group)):
            timestamps.append(base_time)
            timestamps_end.append(base_time + timedelta(minutes=duration_minutes))
            base_time += timedelta(minutes=step_minutes)

    df['timestamp'] = timestamps
    df['timestamp_end'] = timestamps_end
    return df

def convert_to_pm4py_df(df):
    df = df.copy()
    df = df.drop(["concept:name"], axis=1)
    df = df.rename(columns={'name': 'activity', 'trace_id':'case_id', 'user':'org:resource', 'anomaly': 'anomaly'})
    df = df.astype({'activity': str, 'anomaly': str, 'org:resource': str})
    df = format_dataframe(df, case_id='case_id',activity_key='activity', timestamp_key='timestamp')
    df = df.drop(['activity', 'timestamp', 'timestamp_end'], axis=1)
    return df

def convert_and_write_json_to_xes(path_to_json, path_to_xes):
    df = binet_to_df(path_to_json)
    df = assign_sequential_timestamps(df)
    df = convert_to_pm4py_df(df)
    write_xes(df, path_to_xes)

In [29]:
# dataset_names = ["medium", "small", "p2p", "paper"]
dataset_names = ["bpic17-0.3"]
dataset_json_path = r".out/eventlogs/bpic17-0.3-1.json.gz"
dataset_xes_path = r".out/eventlogs/bpic17-0.3-1.xes"
# convert_and_write_json_to_xes(dataset_json_path, dataset_xes_path)

In [30]:
mydf = binet_to_df(dataset_json_path)

In [31]:
mydf.shape

(143518, 16)

In [32]:
convertdf = convert_to_pm4py_df(mydf)

/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/utils.py:132: UserWarning: Some rows of the Pandas data frame have been removed because of empty case IDs, activity labels, or timestamps to ensure the correct functioning of PM4Py's algorithms.
  warnings.warn(
/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/utils.py:137: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[constants.CASE_CONCEPT_NAME] = df[constants.CASE_CONCEPT_NAME].astype(
/home/devashish/anaconda3/envs/ltn/lib/python3.9/site-packages/pm4py/utils.py:141: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: ht

In [33]:
convertdf.head(1)

,anomaly,case_id,org:resource,Accepted,CreditScore,FirstWithdrawalAmount,MonthlyCost,NumberOfTerms,OfferedAmount,Selected,OfferID,case:concept:name,concept:name,time:timestamp,@@index,@@case_index
0,normal,Application_1000086665,User_1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Application_1000086665,A_Create Application,2016-08-03 15:57:21.673000+00:00,0,0


In [34]:
write_xes(convertdf, dataset_xes_path)  

exporting log, completed traces ::   0%|          | 0/6801 [00:00<?, ?it/s]